# STAGE 1: Kaggle T4 Training - 30K Hard Subset

**Goal**: Mài precision từ mAP 80% → 82-86% bằng cách:
- Warm-start từ best.pth (mAP 80%)
- Đóng băng Vision Encoder (Swin-B)
- Train: Cross-Attention + ITM head + Box/Anomaly heads + Pose + XBM Queue
- Strong augmentation (Sim2Real)
- Hard-neg 2 layers (ID-hard + ANCE mining)

**Safety**: Revert nếu mAP < 80%

**Estimated time**: ~2 giờ trên T4 (30K dataset, 3 epochs)

## 1. Setup Environment

In [ ]:
# Clone repo từ GitHub
!git clone https://github.com/Khanhhh239/Model_XVLM_Training.git
%cd Model_XVLM_Training/trainv4

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q albumentations  # Strong augmentation
!pip install -q -e .  # Install package in editable mode

## 2. Download Data & Checkpoint

In [ ]:
# Download best.pth checkpoint (mAP 80%)
# ← BẠN CẦN UPLOAD best.pth LÊN KAGGLE DATASET TRƯỚC!
# Ví dụ: /kaggle/input/xvlm-checkpoints/best.pth

import shutil
from pathlib import Path

# Create directories
Path("data/checkpoints").mkdir(parents=True, exist_ok=True)
Path("data").mkdir(exist_ok=True)

# Copy checkpoint
checkpoint_source = "/kaggle/input/xvlm-checkpoints/best.pth"  # ← ADJUST PATH!
if Path(checkpoint_source).exists():
    shutil.copy(checkpoint_source, "data/checkpoints/best.pth")
    print("✓ Checkpoint copied!")
else:
    print("⚠️ Checkpoint not found! Upload best.pth to Kaggle Dataset first.")

In [ ]:
# Download 30K hard dataset
# ← BẠN CẦN UPLOAD DATA LÊN KAGGLE DATASET!
# Expected files:
#   - train_30k_hard.jsonl
#   - train_30k_hard_vitpose.json
#   - boxes_30k.jsonl
#   - train_webp/ (folder chứa ảnh .webp)

data_source = "/kaggle/input/train-30k-hard"  # ← ADJUST PATH!
if Path(data_source).exists():
    # Symlink hoặc copy data
    !ln -sf {data_source} data/train_30k_hard
    print("✓ Data linked!")
    
    # Verify files
    required_files = [
        "data/train_30k_hard/train_30k_hard.jsonl",
        "data/train_30k_hard/train_30k_hard_vitpose.json",
        "data/train_30k_hard/boxes_30k.jsonl",
    ]
    for f in required_files:
        if Path(f).exists():
            print(f"  ✓ {f}")
        else:
            print(f"  ⚠️ MISSING: {f}")
else:
    print("⚠️ Data not found! Upload train_30k_hard dataset to Kaggle first.")

## 3. Verify Config

In [ ]:
# Load config
import yaml
from pathlib import Path

config_path = "configs/stage1_30k_kaggle_t4.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("📋 Config:")
print(f"  Batch size: {config['train']['batch_size']}")
print(f"  Epochs: {config['optim']['epochs']}")
print(f"  XBM Queue: {config['loss']['xbm_enabled']} (size={config['loss'].get('xbm_size', 0)})")
print(f"  Box head: {config['model']['bbox_enabled']}")
print(f"  Anomaly head: {config['model']['anomaly_enabled']}")
print(f"  Pose: {config['model']['pose_enabled']}")

# Update paths in config if needed
config['data']['manifest'] = 'data/train_30k_hard/train_30k_hard.jsonl'
config['data']['image_root'] = 'data/train_30k_hard/train_webp/'
config['data']['vitpose_json'] = 'data/train_30k_hard/train_30k_hard_vitpose.json'
config['data']['boxes_json'] = 'data/train_30k_hard/boxes_30k.jsonl'
config['model']['checkpoint'] = 'data/checkpoints/best.pth'

# Save updated config
with open('configs/stage1_30k_kaggle_t4_updated.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("\n✓ Config updated and saved to configs/stage1_30k_kaggle_t4_updated.yaml")

## 4. Sanity Check: Overfit One Batch

In [ ]:
# Test training pipeline: loss should drop toward 0 in ~200 steps
!python scripts/train.py \
    --config configs/stage1_30k_kaggle_t4_updated.yaml \
    --init-from data/checkpoints/best.pth \
    --overfit-one-batch

print("\n✓ Sanity check passed! Pipeline is working.")

## 5. TRAIN!

In [ ]:
# Full training with safety net
# - Warm-start từ best.pth
# - Early stop nếu mAP tụt
# - Max 11.5 giờ để tránh Kaggle timeout (12h limit)

!python scripts/train.py \
    --config configs/stage1_30k_kaggle_t4_updated.yaml \
    --init-from data/checkpoints/best.pth \
    --max-hours 11.5

print("\n🎉 Training completed!")

## 6. Evaluate Results

In [ ]:
# Evaluate best checkpoint
!python scripts/evaluate.py \
    --config configs/stage1_30k_kaggle_t4_updated.yaml \
    --ckpt outputs/stage1_30k_t4/best.pth

# Show results
import json
from pathlib import Path

best_ckpt_path = Path("outputs/stage1_30k_t4/best.pth")
if best_ckpt_path.exists():
    import torch
    ckpt = torch.load(best_ckpt_path, map_location='cpu')
    report = ckpt.get('report', {})
    
    print("\n📊 FINAL RESULTS:")
    print(f"  mAP: {report.get('mAP', 'N/A'):.4f}")
    print(f"  R@1: {report.get('R@1', 'N/A'):.4f}")
    print(f"  R@5: {report.get('R@5', 'N/A'):.4f}")
    print(f"  R@10: {report.get('R@10', 'N/A'):.4f}")
    print(f"  MRR: {report.get('MRR', 'N/A'):.4f}")
    
    # Safety check
    baseline_map = 0.80
    final_map = report.get('mAP', 0.0)
    if final_map >= baseline_map:
        gain = (final_map - baseline_map) * 100
        print(f"\n✅ SUCCESS! mAP improved by +{gain:.2f}% (from 80.00% to {final_map*100:.2f}%)")
    else:
        loss = (baseline_map - final_map) * 100
        print(f"\n⚠️ WARNING: mAP dropped by -{loss:.2f}% (from 80.00% to {final_map*100:.2f}%)")
        print("   → Safety net should have reverted. Check logs.")
else:
    print("⚠️ Best checkpoint not found!")

## 7. Save Checkpoint for Submission

In [ ]:
# Copy best checkpoint to output directory for easy download
import shutil

output_dir = Path("/kaggle/working")
best_ckpt = Path("outputs/stage1_30k_t4/best.pth")

if best_ckpt.exists():
    shutil.copy(best_ckpt, output_dir / "stage1_best.pth")
    print(f"✓ Checkpoint saved to {output_dir / 'stage1_best.pth'}")
    print(f"  Size: {best_ckpt.stat().st_size / (1024**2):.1f} MB")
else:
    print("⚠️ No checkpoint to save!")

## 8. Training Logs & Analysis

In [ ]:
# Plot loss curves (if logs available)
import matplotlib.pyplot as plt
import re

log_file = Path("outputs/stage1_30k_t4/train.log")
if log_file.exists():
    with open(log_file, 'r') as f:
        lines = f.readlines()
    
    # Parse losses
    steps, losses_itc, losses_itm, losses_box, losses_anomaly = [], [], [], [], []
    for line in lines:
        match = re.search(r's(\d+) loss=([\d.]+) itc=([\d.]+) itm=([\d.]+)', line)
        if match:
            steps.append(int(match.group(1)))
            losses_itc.append(float(match.group(3)))
            losses_itm.append(float(match.group(4)))
    
    if steps:
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        ax.plot(steps, losses_itc, label='ITC', alpha=0.7)
        ax.plot(steps, losses_itm, label='ITM', alpha=0.7)
        ax.set_xlabel('Step')
        ax.set_ylabel('Loss')
        ax.set_title('Training Losses')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('loss_curves.png', dpi=150)
        plt.show()
        print("✓ Loss curves saved to loss_curves.png")
else:
    print("⚠️ Log file not found")

## 9. Next Steps

- **If mAP improved**: Download checkpoint và tiếp tục với STAGE 2 (unfreeze vision encoder)
- **If mAP dropped**: Review logs, adjust hyperparameters (LR, loss weights)
- **Hard-neg mining**: Implement ANCE cross-ID mining script (mỗi epoch)
- **Ensemble**: Kết hợp với SigLIP retriever (như architecture plan)